# Gray-Scott WELL-like End-to-End Example

This notebook runs Gray-Scott directly with the `Simulator` API, saves a WELL-like `.npz`, and creates visualizations (inline animation + GIF).

In [ ]:
from pathlib import Path

import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

from autosim.simulations import GrayScott
from autosim.utils import plot_spatiotemporal_video

In [ ]:
# Run one WELL-like Gray-Scott trajectory directly from the Simulator API.
project_root = Path.cwd()
output_path = project_root / "examples/gray_scott_well_like_gliders.npz"
gif_path = project_root / "examples/gray_scott_well_like_gliders.gif"

sim = GrayScott(
    pattern="gliders",
    return_timeseries=True,
    log_level="warning",
    n=128,
    T=10000.0,
    dt=1.0,
    snapshot_dt=10.0,
    initial_condition="gaussian",
    random_seed=0,
)

x = sim.sample_inputs(1, random_seed=0)
y, x_valid = sim.forward_batch(x, allow_failures=False)
y, x_valid

n_channels = 2
n_steps = y.shape[1] // (n_channels * sim.n * sim.n)
channel_time_xy = y.reshape(1, n_channels, n_steps, sim.n, sim.n)[0].cpu().numpy()
data = np.transpose(channel_time_xy, (1, 2, 3, 0)).astype(np.float32)  # (time, x, y, channels)

output_path.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    output_path,
    data=data,
    constant_scalars=x_valid[0].cpu().numpy().astype(np.float32),
    param_names=np.array(sim.param_names),
    pattern=np.array("gliders"),
    initial_condition=np.array("gaussian"),
    n=np.array(sim.n),
    t_max=np.array(sim.T),
    dt=np.array(sim.dt),
    snapshot_dt=np.array(sim.snapshot_dt),
    random_seed=np.array(0),
)

print("saved:", output_path)
print("data shape:", data.shape)
print("params:", dict(zip(sim.param_names, x_valid[0].tolist(), strict=False)))

In [ ]:
# Reload and inspect saved trajectory metadata.
arr = np.load(output_path, allow_pickle=False)
data = arr["data"]

print("data shape:", data.shape)
print("param names:", arr["param_names"])
print("constant scalars:", arr["constant_scalars"])

In [ ]:
# Build inline animation from saved data.
data_batched = torch.from_numpy(data).unsqueeze(0)  # [batch, time, x, y, channels]
anim = plot_spatiotemporal_video(data_batched, batch_idx=0)

In [ ]:
from IPython.display import HTML

HTML(anim.to_jshtml())

In [ ]:
# Create a GIF for species A (channel 0) and display it.
u_frames = data[::5, :, :, 0]
vmin = float(u_frames.min())
vmax = float(u_frames.max())
if vmax <= vmin:
    vmax = vmin + 1e-6

cmap = plt.get_cmap("viridis")
images = []
for frame in u_frames:
    norm = (frame - vmin) / (vmax - vmin)
    rgba = (cmap(np.clip(norm, 0.0, 1.0))[:, :, :3] * 255).astype(np.uint8)
    images.append(Image.fromarray(rgba))

images[0].save(
    gif_path,
    save_all=True,
    append_images=images[1:],
    duration=70,
    loop=0,
)
print("saved:", gif_path)

In [ ]:
from IPython.display import Image

Image(filename=str(gif_path))

In [ ]:
data.shape

In [ ]:
plt.plot(data[:,0,0,:])